# Notebook 4.3 - AST-Tiny Baseline

Notebook này chạy riêng model `ast_tiny` với 5 seed `42-46` để lấy số liệu bảng so sánh.


In [1]:
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

from src.baseline_experiment import run_baseline_suite
from src.baseline_protocol import (
    DEFAULT_BASELINE_SEEDS,
    build_output_dir,
    ensure_seed_completeness,
    filter_runs_for_run,
    make_fair_train_config,
)

data_dir = repo_root / 'data' / 'features' / 'mel'
base_output_dir = repo_root / 'data' / 'models' / 'baselines'

RUN_ID = 'paper_v1'
output_dir = build_output_dir(base_output_dir=base_output_dir, run_id=RUN_ID)

print(f'Repo root: {repo_root}')
print(f'Data dir: {data_dir}')
print(f'Output dir: {output_dir}')
print(f'Run ID: {RUN_ID}')


Repo root: /home/anhcbt/extend/workspace/convmixer_model
Data dir: /home/anhcbt/extend/workspace/convmixer_model/data/features/mel
Output dir: /home/anhcbt/extend/workspace/convmixer_model/data/models/baselines/paper_v1
Run ID: paper_v1


/home/anhcbt/extend/workspace/convmixer_model/src/models/ast_official_models.py:204: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast()
/home/anhcbt/extend/workspace/convmixer_model/.venv/lib/python3.13/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


## Protocol

- Model: `ast_tiny`
- Seeds: `42, 43, 44, 45, 46`
- Mục tiêu: lấy số liệu cho bảng, không vẽ biểu đồ.


In [2]:
selected_model = 'ast_tiny'
model_label = 'AST-Tiny'
selected_seeds = DEFAULT_BASELINE_SEEDS
USE_CACHED_RESULTS = False  # True: chỉ đọc CSV trong đúng run_id, không train

config = make_fair_train_config(
    data_dir=data_dir,
    base_output_dir=base_output_dir,
    run_id=RUN_ID,
    model_names=(selected_model,),
    seeds=selected_seeds,
    include_existing_runs=True,
    skip_completed_runs=True,
)

config


TrainConfig(data_dir='/home/anhcbt/extend/workspace/convmixer_model/data/features/mel', output_dir='/home/anhcbt/extend/workspace/convmixer_model/data/models/baselines/paper_v1', run_id='paper_v1', data_version='paper_v1', model_names=('ast_tiny',), seeds=(42, 43, 44, 45, 46), split_seed=42, train_ratio=0.75, val_ratio=0.15, batch_size=32, num_workers=4, num_epochs=25, lr=0.001, weight_decay=0.0001, patience=5, min_delta=0.02, scheduler_factor=0.2, scheduler_patience=3, scheduler_min_lr=1e-06, ast_official_model_size='tiny224', ast_official_fstride=10, ast_official_tstride=10, ast_official_input_fdim=128, ast_official_input_tdim=32, ast_official_imagenet_pretrain=False, ast_official_audioset_pretrain=False, ast_official_verbose=True, ast_official_auto_input_shape=True, ast_official_auto_norm_from_train=True, ast_official_norm_mean=None, ast_official_norm_std=None, ast_official_lr=0.001, ast_official_weight_decay=0.0001, use_model_specific_hparams=False, include_existing_runs=True, skip

In [3]:
def build_single_summary(runs: pd.DataFrame) -> pd.DataFrame:
    if runs.empty:
        return pd.DataFrame()
    return (
        runs.groupby('model', as_index=False)
        .agg(
            n_runs=('seed', 'nunique'),
            params=('params', 'mean'),
            test_accuracy_mean=('test_accuracy', 'mean'),
            test_accuracy_std=('test_accuracy', 'std'),
            macro_f1_mean=('macro_f1', 'mean'),
            macro_f1_std=('macro_f1', 'std'),
            weighted_f1_mean=('weighted_f1', 'mean'),
            weighted_f1_std=('weighted_f1', 'std'),
            train_seconds_mean=('train_seconds', 'mean'),
            infer_seconds_mean=('infer_seconds', 'mean'),
        )
        .fillna(0.0)
    )

runs_cache_path = output_dir / 'baseline_runs.csv'

def _job_paths(seed: int) -> dict:
    return {
        'ckpt': output_dir / f'{selected_model}_seed{seed}_best.pth',
        'report': output_dir / f'{selected_model}_seed{seed}_report.json',
        'done': output_dir / f'{selected_model}_seed{seed}_done.json',
    }

def _read_runs_cache() -> pd.DataFrame:
    if runs_cache_path.exists():
        return pd.read_csv(runs_cache_path)
    return pd.DataFrame()

def _has_run_row(runs_df: pd.DataFrame, seed: int) -> bool:
    if runs_df.empty:
        return False
    required_cols = {'model', 'seed', 'run_id'}
    if not required_cols.issubset(set(runs_df.columns)):
        return False
    mask = (
        (runs_df['model'].astype(str) == str(selected_model))
        & (runs_df['seed'].astype(int) == int(seed))
        & (runs_df['run_id'].astype(str) == str(RUN_ID))
    )
    return bool(mask.any())

def _append_or_replace_run_row(new_row: dict) -> None:
    runs_df = _read_runs_cache()
    if not runs_df.empty and {'model', 'seed', 'run_id'}.issubset(set(runs_df.columns)):
        keep_mask = ~(
            (runs_df['model'].astype(str) == str(new_row['model']))
            & (runs_df['seed'].astype(int) == int(new_row['seed']))
            & (runs_df['run_id'].astype(str) == str(new_row['run_id']))
        )
        runs_df = runs_df[keep_mask].copy()
    runs_df = pd.concat([runs_df, pd.DataFrame([new_row])], ignore_index=True)
    runs_df.to_csv(runs_cache_path, index=False)

def _recover_row_from_report(seed: int) -> dict:
    paths = _job_paths(seed)
    with open(paths['report'], 'r', encoding='utf-8') as f:
        report = json.load(f)
    acc = float(report.get('accuracy', np.nan))
    if np.isfinite(acc) and acc <= 1.0:
        acc = acc * 100.0
    return {
        'model': selected_model,
        'seed': int(seed),
        'run_id': RUN_ID,
        'test_accuracy': acc,
        'macro_f1': float(report.get('macro avg', {}).get('f1-score', np.nan)),
        'weighted_f1': float(report.get('weighted avg', {}).get('f1-score', np.nan)),
        'checkpoint_path': str(paths['ckpt']),
        'report_path': str(paths['report']),
    }

def _write_done_marker(seed: int) -> None:
    paths = _job_paths(seed)
    payload = {
        'model': selected_model,
        'seed': int(seed),
        'run_id': RUN_ID,
        'checkpoint_path': str(paths['ckpt']),
        'report_path': str(paths['report']),
    }
    with open(paths['done'], 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

if USE_CACHED_RESULTS:
    print('[MODE] Cached: đọc CSV hiện có trong run_id, không train lại.')
    if not runs_cache_path.exists():
        raise FileNotFoundError(f'Không tìm thấy cache: {runs_cache_path}')
    runs_df = pd.read_csv(runs_cache_path)
    history_store = {}
    metadata = {
        'mode': 'cached',
        'run_id': RUN_ID,
        'runs_csv': str(runs_cache_path),
    }
else:
    print('[MODE] Incremental train: quét file trước, chỉ train seed còn thiếu.')
    history_store = {}
    job_metadata = []
    skipped_by_file = 0
    recovered_rows = 0

    total_jobs = len(selected_seeds)
    for job_idx, seed in enumerate(selected_seeds, start=1):
        paths = _job_paths(seed)
        ckpt_exists = paths['ckpt'].exists()
        report_exists = paths['report'].exists()
        file_done = ckpt_exists and report_exists
        runs_cache_df = _read_runs_cache()
        has_row = _has_run_row(runs_cache_df, seed=seed)

        if file_done and has_row:
            skipped_by_file += 1
            print(f'[SKIP-FILE] [{job_idx}/{total_jobs}] model={selected_model}, seed={seed} (file+row đã có)')
            continue

        if file_done and not has_row:
            _append_or_replace_run_row(_recover_row_from_report(seed=seed))
            recovered_rows += 1
            skipped_by_file += 1
            print(f'[RECOVER] [{job_idx}/{total_jobs}] model={selected_model}, seed={seed} (khôi phục vào CSV từ report)')
            continue

        print(f'[TRAIN] [{job_idx}/{total_jobs}] model={selected_model}, seed={seed}')
        seed_config = make_fair_train_config(
            data_dir=data_dir,
            base_output_dir=base_output_dir,
            run_id=RUN_ID,
            model_names=(selected_model,),
            seeds=(int(seed),),
            include_existing_runs=True,
            skip_completed_runs=True,
        )
        _, _, seed_history, one_meta = run_baseline_suite(seed_config)
        history_store.update(seed_history)
        job_metadata.append(one_meta)
        _write_done_marker(seed=seed)

    runs_df = _read_runs_cache()

    metadata = {
        'mode': 'incremental_train',
        'run_id': RUN_ID,
        'runs_csv': str(runs_cache_path),
        'jobs_total': total_jobs,
        'jobs_trained': int(sum(m.get('trained_count', 0) for m in job_metadata)),
        'jobs_skipped': int(sum(m.get('skipped_count', 0) for m in job_metadata)),
        'jobs_skipped_by_file': skipped_by_file,
        'jobs_recovered_rows': recovered_rows,
    }

runs_df = filter_runs_for_run(
    runs_df=runs_df,
    model_names=(selected_model,),
    run_id=RUN_ID,
)
ensure_seed_completeness(
    runs_df=runs_df,
    model_names=(selected_model,),
    seeds=selected_seeds,
)
summary_df = build_single_summary(runs_df)

runs_selected_path = output_dir / f'baseline_runs_{selected_model}.csv'
summary_selected_path = output_dir / f'baseline_summary_{selected_model}.csv'

runs_df.to_csv(runs_selected_path, index=False)
summary_df.to_csv(summary_selected_path, index=False)

print('Metadata:')
print(metadata)
print('Saved:', runs_selected_path)
print('Saved:', summary_selected_path)


[MODE] Train: train phần thiếu trong run_id hiện tại.
[AST] Using official AST settings: input_fdim=128, input_tdim=32, norm_mean=0.308112, norm_std=0.275267, lr=0.001, wd=0.0001
[INFO] Loaded 0 existing runs from /home/anhcbt/extend/workspace/convmixer_model/data/models/baselines/paper_v1/baseline_runs.csv for run_id=paper_v1
---------------AST Model Summary---------------
ImageNet pretraining: False, AudioSet pretraining: False
frequncey stride=10, time stride=10
number of patches=24
[START] model=ast_tiny seed=42 params=5782365 device=cpu lr=0.001 wd=0.0001
[ast_tiny|seed42] Epoch 1/25 | train_loss=2.3371 | val_loss=2.2389 | val_acc=17.59% | lr=0.001000->0.001000 | improved
[ast_tiny|seed42] Epoch 2/25 | train_loss=2.2120 | val_loss=2.1258 | val_acc=25.71% | lr=0.001000->0.001000 | improved
[ast_tiny|seed42] Epoch 3/25 | train_loss=2.1482 | val_loss=2.0097 | val_acc=27.67% | lr=0.001000->0.001000 | improved
[ast_tiny|seed42] Epoch 4/25 | train_loss=2.0995 | val_loss=1.9171 | val_acc

KeyboardInterrupt: 

In [ ]:
display_cols = [
    'model', 'seed', 'params', 'best_epoch', 'test_accuracy',
    'macro_f1', 'weighted_f1', 'infer_seconds', 'train_seconds'
]

runs_df[display_cols].sort_values('seed')


In [ ]:
params_m = runs_df['params'].mean() / 1_000_000

acc_mean = runs_df['test_accuracy'].mean()
acc_std = runs_df['test_accuracy'].std(ddof=1)
macro_f1_mean = runs_df['macro_f1'].mean()
macro_f1_std = runs_df['macro_f1'].std(ddof=1)
weighted_f1_mean = runs_df['weighted_f1'].mean()
weighted_f1_std = runs_df['weighted_f1'].std(ddof=1)
infer_mean = runs_df['infer_seconds'].mean()
infer_std = runs_df['infer_seconds'].std(ddof=1)
train_mean = runs_df['train_seconds'].mean()
train_std = runs_df['train_seconds'].std(ddof=1)

latex_row = (
    f"{model_label} & {params_m:.2f} & "
    f"{acc_mean:.2f}$\\pm${acc_std:.2f} & "
    f"{macro_f1_mean:.4f}$\\pm${macro_f1_std:.4f} & "
    f"{weighted_f1_mean:.4f}$\\pm${weighted_f1_std:.4f} & "
    f"{infer_mean:.4f}$\\pm${infer_std:.4f} & "
    f"{train_mean:.1f}$\\pm${train_std:.1f} \\\\"
)

result_df = pd.DataFrame([
    {
        'model_label': model_label,
        'params_m': params_m,
        'test_accuracy_mean': acc_mean,
        'test_accuracy_std': acc_std,
        'macro_f1_mean': macro_f1_mean,
        'macro_f1_std': macro_f1_std,
        'weighted_f1_mean': weighted_f1_mean,
        'weighted_f1_std': weighted_f1_std,
        'infer_seconds_mean': infer_mean,
        'infer_seconds_std': infer_std,
        'train_seconds_mean': train_mean,
        'train_seconds_std': train_std,
    }
])

metrics_path = output_dir / f'baseline_table_metrics_{selected_model}.csv'
result_df.to_csv(metrics_path, index=False)

print(latex_row)
print('Saved:', metrics_path)
result_df
